# 진행 상황 보기 · stream

강의 오후 ③ 앞부분 (Academy 모듈 3 · L1 Streaming)

| 칸 | 보여 주는 것 |
| --- | --- |
| ① invoke | 다 돌 때까지 아무것도 안 보인다. 몇 초 뒤에 한꺼번에 |
| ② stream · updates | 노드가 끝날 때마다, 그 노드가 바꾼 것만 받는다 |
| ③ stream · values | 노드가 끝날 때마다 State 전체를 받는다 |
| ④ 토큰 | 모델이 쓰는 글자를 받는 대로 보여 준다. 노드 안에서 `config.writer` 로 흘려보내고 `streamMode: "custom"` 으로 받는다 |
| ⑤ 모드 여러 개 | `streamMode: ["values", "custom"]` 처럼 배열로 주면 두 모드의 값이 `[모드, 값]` 으로 한 루프에 섞여 온다 |

모델은 실습과 같은 OpenRouter 의 `qwen/qwen3-32b` 이고, 접속 정보는 `lab/.env` 에서 읽습니다.

In [72]:
import "npm:@langchain/core@1.2.14";
import { Annotation, StateGraph, START, END, MemorySaver } from "npm:@langchain/langgraph@1.4.18";

// 접속 정보: 실습과 같은 lab/.env 를 읽는다 (OpenRouter · qwen/qwen3-32b). 키는 화면에 찍지 않는다
const ENV: Record<string, string> = {};
try {
  for (const line of Deno.readTextFileSync("../lab/.env").split("\n")) {
    const m = line.match(/^\s*([A-Z_]+)\s*=\s*(.*)\s*$/);
    if (m) ENV[m[1]] = m[2];
  }
} catch { /* lab/.env 가 없으면 아래 기본값 */ }
const BASE_URL = ENV.LLM_BASE_URL ?? "https://openrouter.ai/api/v1";
const API_KEY  = ENV.LLM_API_KEY  ?? "";
const MODEL    = ENV.LLM_MODEL    ?? "qwen/qwen3-32b";
const OPTIONS  = { temperature: 0 };   // 시연이 매번 같게
console.log({ BASE_URL, MODEL, 키: API_KEY ? "있음" : "없음 — lab/.env 에 LLM_API_KEY 를 넣는다" });

// LangSmith (강사 시연용): demo/.env 에 키가 있으면 실행 기록(trace)이 LangSmith 에 남는다. 실습은 lab/.env 를 써서 남지 않는다
import { traceable } from "npm:langsmith@0.11.1/traceable";
try {
  for (const line of Deno.readTextFileSync(".env").split("\n")) {
    const m = line.match(/^\s*(LANGSMITH_[A-Z_]+)\s*=\s*(?:"([^"]*)"|([^#\s]*))/);   // "따옴표 값" 은 그대로, 아니면 # 주석 앞까지
    const value = m?.[2] ?? m?.[3];
    if (m && value) Deno.env.set(m[1], value);
  }
} catch { /* demo/.env 가 없으면 기록하지 않는다 */ }
if (!Deno.env.get("LANGSMITH_API_KEY")) Deno.env.set("LANGSMITH_TRACING", "false");   // 키가 없으면 끈다
// 모델을 부르는 함수를 감싸면 trace 에 "모델 호출" 한 칸으로 남는다. 기록을 안 켜면 그냥 원래 함수로 돈다
const asLLM = <F extends (...a: any[]) => Promise<any>>(fn: F, name: string) => traceable(fn, {
  name, run_type: "llm",
  processInputs: (i: any) => ({ messages: i.args ? i.args[0] : i.input }),
  processOutputs: (o: any) => ({ choices: [{ message: typeof o.outputs === "string" ? { role: "assistant", content: o.outputs } : o }] }),
});
console.log({ LangSmith: Deno.env.get("LANGSMITH_TRACING") === "true" && Deno.env.get("LANGSMITH_API_KEY") ? `기록함 · project ${Deno.env.get("LANGSMITH_PROJECT") ?? "default"}` : "기록 안 함 (demo/.env)" });

// 그래프 그림: 노드·엣지 이름을 mermaid.ink 로 보내 PNG 로 받아 셀 출력에 띄운다. 받지 못하면 mermaid 글자로 찍는다
async function showGraph(graph: any, options: { xray?: boolean } = {}) {
  const drawable = await graph.getGraphAsync(options);   // xray: true 면 subgraph 안의 노드까지 그린다
  try {
    const png = await drawable.drawMermaidPng();
    await Deno.jupyter.display(Deno.jupyter.image(new Uint8Array(await png.arrayBuffer())));
  } catch {
    console.log(drawable.drawMermaid());
  }
}

{
  BASE_URL: "https://openrouter.ai/api/v1",
  MODEL: "qwen/qwen3-32b",
  "키": "있음"
}
{ LangSmith: "기록함 · project AI Agent " }


## 준비: 오후 ① 의 튜터 Agent

모델 부르기(`chat()`), 사전 도구(`lookup_word`), 그래프는 오후 ① · ② 에서 쓴 것 그대로입니다. tutor 가 모델을 부르고, 도구를 원하면 tools 로 갔다가 다시 tutor 로 돌아옵니다. 이번에도 `compile()` 하기 전의 `builder` 까지만 만들어 둡니다. 같은 그래프를 여러 방식으로 compile 하려고요.

In [73]:
// 오전에 만든 chat() 그대로 — 다시 보낼 칸(role · content · tool_calls)만 돌려준다
async function callModel(messages: any[], tools?: any[]) {
  const res = await fetch(`${BASE_URL}/chat/completions`, {
    method: "POST",
    headers: { "Content-Type": "application/json", Authorization: `Bearer ${API_KEY}` },
    body: JSON.stringify({ model: MODEL, messages, tools, ...OPTIONS }),
  });
  const { role, content, tool_calls } = (await res.json()).choices[0].message;
  return { role, content: content?.trim() ?? null, tool_calls };
}
const chat = asLLM(callModel, "chat");   // LangSmith 에 모델 호출로 남긴다

// 도구 하나: 단어 뜻과 쉬운 말 찾기 (작은 사전이라 시연이 매번 같다)
const DICT: Record<string, { 뜻: string; 쉬운말: string }> = {
  ubiquitous: { 뜻: "어디에나 있는", 쉬운말: "everywhere" },
  mitigate:   { 뜻: "완화하다, 줄이다", 쉬운말: "reduce" },
  meticulous: { 뜻: "꼼꼼한", 쉬운말: "careful" },
  detrimental:{ 뜻: "해로운", 쉬운말: "harmful" },
};
const tools = [{
  type: "function",
  function: {
    name: "lookup_word",
    description: "영어 단어 하나의 뜻과, 바꿔 쓸 쉬운 말을 사전에서 찾는다",
    parameters: { type: "object", properties: { word: { type: "string", description: "찾을 영어 단어 하나" } }, required: ["word"] },
  },
}];
function lookupWord(word: string) {
  const w = word.toLowerCase().trim();
  return DICT[w] ? { word: w, ...DICT[w] } : { word: w, error: "사전에 없는 단어" };
}

const SYSTEM = { role: "system", content: "너는 IELTS 쓰기를 돕는 영어 튜터다. 단어 뜻은 추측하지 말고 lookup_word 로 확인한다. 한국어로 짧게 답한다." };
const ask = (q: string) => ({ messages: [SYSTEM, { role: "user", content: q }] });
function show(out: any) {
  const last = out.messages.at(-1);
  console.log(`messages ${out.messages.length}개 · 마지막: ${last.role}`);
  console.log(last.tool_calls?.length ? `  도구 요청 ${JSON.stringify(last.tool_calls.map((c: any) => c.function))}` : `  ${last.content}`);
}

In [74]:
// 오후 ① 의 튜터 Agent 그대로: tutor ⇢ tools → tutor
const State = Annotation.Root({
  messages: Annotation<any[]>({ reducer: (prev, next) => [...prev, ...next], default: () => [] }),
});
async function tutor(state: typeof State.State) {
  return { messages: [await chat(state.messages, tools)] };
}
function toolsNode(state: typeof State.State) {
  const calls = state.messages.at(-1).tool_calls;
  console.log(`▶ tools   ${calls.map((c: any) => c.function.arguments).join(" · ")}`);
  return { messages: calls.map((c: any) => ({
    role: "tool", tool_call_id: c.id, content: JSON.stringify(lookupWord(JSON.parse(c.function.arguments).word)),
  })) };
}
function route(state: typeof State.State) {
  return state.messages.at(-1).tool_calls?.length ? "tools" : END;
}
const builder = new StateGraph(State)
  .addNode("tutor", tutor)
  .addNode("tools", toolsNode)
  .addEdge(START, "tutor")
  .addConditionalEdges("tutor", route, ["tools", END])
  .addEdge("tools", "tutor");

## ① invoke 는 다 돌 때까지 기다린다

지금까지는 `invoke` 로 불렀습니다. invoke 는 그래프가 END 에 닿을 때까지 기다렸다가 마지막 State 하나를 돌려줍니다. 단어 두 개를 찾아서 문장을 고쳐 달라고 해 보겠습니다. 걸린 시간도 재 봅니다.

In [75]:
// ① invoke: 다 돌 때까지 아무것도 안 보인다
const agent = builder.compile();
await showGraph(agent);
const Q = "이 문장의 ubiquitous 랑 mitigate 를 사전에서 찾아서, 쉬운 말로 바꾼 영어 문장을 만들어 줘: Smartphones are ubiquitous, and schools try to mitigate their effects.";
const sec = (t0: number) => `${((Date.now() - t0) / 1000).toFixed(1)}초`;
let t0 = Date.now();
show(await agent.invoke(ask(Q)));
console.log(`→ ${sec(t0)} 뒤에 한꺼번에 나왔다`);

▶ tools   {"word": "ubiquitous"} · {"word": "mitigate"}
messages 6개 · 마지막: assistant
  Smartphones are everywhere, and schools try to reduce their effects.
→ 12.8초 뒤에 한꺼번에 나왔다


**돌려 보면** 십몇 초 동안 아무것도 안 보이다가 답이 한꺼번에 나옵니다. 중간에 찍힌 `▶ tools` 는 노드 안의 console.log 입니다. 그래프를 부른 쪽, 예를 들어 학생이 보는 화면은 그동안 아무것도 받지 못합니다.

## ② stream · updates — 노드가 바꾼 것만

`invoke` 대신 `stream` 으로 부르면, 노드가 하나 끝날 때마다 값을 하나씩 받습니다. `for await` 로 하나씩 꺼냅니다.

무엇을 받을지는 `streamMode` 로 고릅니다. 먼저 `"updates"`. 받는 값은 `{ 노드 이름: 그 노드가 돌려준 값 }` 모양입니다.

In [76]:
// ② stream · updates: 노드가 끝날 때마다, 그 노드가 바꾼 것만 받는다
// 메시지를 보기 좋게 찍는다 (LangChain 의 pretty_print 처럼)
const TITLE: Record<string, string> = {
  system: "System Message",
  user: "Human Message",
  assistant: "Ai Message",
  tool: "Tool Message",
};
function pretty(m: any) {
  const title = ` ${TITLE[m.role]} `;
  const left = Math.floor((60 - title.length) / 2);
  console.log("=".repeat(left) + title + "=".repeat(60 - title.length - left));
  if (m.tool_calls?.length) {
    const calls = m.tool_calls.map((c: any) => `${c.function.name}(${c.function.arguments})`);
    console.log("Tool Calls:", calls.join(", "));
  } else console.log(m.content);
}

t0 = Date.now();

for await (const chunk of await agent.stream(ask(Q), {
  streamMode: "updates",
})) {
  for (const [node, update] of Object.entries(chunk) as [string, any][]) {
    console.log(`\n${sec(t0)} · 노드명:[${node}] 값: ${update}`);
    for (const m of update.messages) pretty(m);
  }
}

▶ tools   {"word": "ubiquitous"} · {"word": "mitigate"}

7.1초 · 노드명:[tutor] 값: [object Object]
======================== Ai Message ========================
Tool Calls: lookup_word({"word": "ubiquitous"}), lookup_word({"word": "mitigate"})

7.1초 · 노드명:[tools] 값: [object Object]
======================= Tool Message =======================
{"word":"ubiquitous","뜻":"어디에나 있는","쉬운말":"everywhere"}
======================= Tool Message =======================
{"word":"mitigate","뜻":"완화하다, 줄이다","쉬운말":"reduce"}

14.8초 · 노드명:[tutor] 값: [object Object]
======================== Ai Message ========================
Smartphones are everywhere, and schools try to reduce their effects.


**돌려 보면** 값이 세 번 옵니다. 받은 값은 `{ 노드 이름: 그 노드가 return 한 값 }` 이라서, 노드 이름과 그 노드가 돌려준 메시지만 찍었습니다.

| 언제 | 노드 | 돌려준 메시지 |
| --- | --- | --- |
| 6~8초쯤 | tutor | Ai Message 하나: 도구 요청 두 개 |
| 곧바로 | tools | Tool Message 두 개: 사전 결과 |
| 몇 초 뒤 | tutor | Ai Message 하나: 답 |

tools 가 돌려준 메시지가 2개뿐인 건 그 노드가 돌려준 것만 왔기 때문입니다. State 의 messages 전체가 아닙니다. 다 돌기 전에 중간 과정이 하나씩 들어옵니다.

`▶ tools` 줄이 맨 위에 먼저 찍히는 건, 그래프가 값을 내보내고 나서 우리가 꺼내 가기를 기다리지 않고 바로 다음 노드를 시작하기 때문입니다.

## ③ stream · values — State 전체

`"values"` 는 노드가 끝날 때마다 그때의 **State 전체**를 줍니다. 처음 넣은 입력만 있는 State 가 먼저 나옵니다.

In [77]:
// ③ stream · values: 노드가 끝날 때마다 State 전체를 받는다. 처음 넣은 입력부터 나온다
// system 메시지는 매번 같으니 빼고 찍는다
for await (const state of await agent.stream(ask(Q), {
  streamMode: "values",
})) {
  for (const m of state.messages) if (m.role !== "system") pretty(m);
  console.log("-".repeat(60));
}

====================== Human Message =======================
이 문장의 ubiquitous 랑 mitigate 를 사전에서 찾아서, 쉬운 말로 바꾼 영어 문장을 만들어 줘: Smartphones are ubiquitous, and schools try to mitigate their effects.
------------------------------------------------------------
▶ tools   {"word": "ubiquitous"} · {"word": "mitigate"}
====================== Human Message =======================
이 문장의 ubiquitous 랑 mitigate 를 사전에서 찾아서, 쉬운 말로 바꾼 영어 문장을 만들어 줘: Smartphones are ubiquitous, and schools try to mitigate their effects.
======================== Ai Message ========================
Tool Calls: lookup_word({"word": "ubiquitous"}), lookup_word({"word": "mitigate"})
------------------------------------------------------------
====================== Human Message =======================
이 문장의 ubiquitous 랑 mitigate 를 사전에서 찾아서, 쉬운 말로 바꾼 영어 문장을 만들어 줘: Smartphones are ubiquitous, and schools try to mitigate their effects.
======================== Ai Message ========================
Tool Calls: lookup_word({"word":

**돌려 보면** `-----` 줄로 나뉜 State 가 네 개 옵니다. State 마다 그때의 messages 전체가 나옵니다. 노드 이름은 없습니다. (system 메시지는 매번 같아서 찍지 않았습니다.)

| State | messages |
| --- | --- |
| 1 | Human (처음 넣은 입력) |
| 2 | + Ai 의 도구 요청 |
| 3 | + Tool 두 개 |
| 4 | + Ai 의 답 |

앞의 메시지가 State 마다 다시 나옵니다. 같은 실행을 updates 는 "무엇이 바뀌었나" 로, values 는 "지금 State 가 어떤가" 로 보여 줍니다.

화면에 진행 상황을 띄우려면 updates, 지금 State 를 통째로 그리려면 values 가 편합니다.

## ④ 토큰 — 글자를 받는 대로

updates 로도 답은 tutor 가 다 끝나야 옵니다. ChatGPT 처럼 글자가 써지는 대로 보여 주려면, 모델에게서 답을 **토큰 단위로** 받아야 합니다.

chat API 는 `stream: true` 를 주면 답을 토큰 단위로 나눠 보냅니다. 이 형식을 SSE(Server-Sent Events)라고 합니다. 받는 코드를 보기 전에, 실제로 무엇이 오는지 먼저 들여다보겠습니다. `peekSSE` 는 받은 줄을 그대로 찍어 보는 함수입니다.

In [78]:
// SSE 들여다보기: stream: true 로 부르면 응답이 실제로 어떤 줄로 오는지 찍어 본다
async function peekSSE(messages: any[], tools?: any[], max = 12) {
  const res = await fetch(`${BASE_URL}/chat/completions`, {
    method: "POST",
    headers: {
      "Content-Type": "application/json",
      Authorization: `Bearer ${API_KEY}`,
    },
    body: JSON.stringify({
      model: MODEL,
      messages,
      tools,
      ...OPTIONS,
      stream: true,
    }),
  });

  let pieces = 0,
    shown = 0,
    thinking = 0,
    cut = 0,
    first = true;

  for await (const bytes of res.body!.pipeThrough(new TextDecoderStream())) {
    pieces++; // 네트워크로 받은 덩어리 하나
    for (const line of bytes.split("\n")) {
      if (!line.startsWith("data: ")) continue; // 빈 줄, ": OPENROUTER PROCESSING" 같은 주석 줄
      if (line === "data: [DONE]") continue; // 끝 표시

      // 덩어리 경계에서 잘린 줄은 JSON 으로 못 읽는다. 들여다보기에서는 세기만 한다
      let json;
      try {
        json = JSON.parse(line.slice(6));
      } catch {
        cut++;
        continue;
      }

      // qwen3 가 속으로 생각하는 줄(reasoning)은 우리가 쓰지 않는다. 세기만 한다
      const { role, reasoning, reasoning_details, ...delta } =
        json.choices?.[0]?.delta ?? {};
      if (reasoning) {
        thinking++;
        continue;
      }

      // 한 줄 전체는 이렇게 생겼다 (처음 한 번만 찍는다)
      if (first) {
        console.log(line.slice(0, 160) + " …");
        first = false;
      }
      if (shown++ < max) console.log("  delta", JSON.stringify(delta));
    }
  }

  if (shown > max) console.log(`  … delta ${shown - max}줄 더`);
  console.log("data: [DONE]");
  console.log(
    `네트워크 덩어리 ${pieces}개 · 생각(reasoning) ${thinking}줄은 뺐다 · 덩어리 경계에서 잘린 줄 ${cut}개`,
  );
}

In [79]:
// ㉠ 도구를 부를 때: 단어 두 개 → 도구 요청 두 개 (index 0, 1)
await peekSSE(ask(Q).messages, tools);

data: {"id":"gen-1793097986-uDL9u7hsOjaM2r0AF5Yy","object":"chat.completion.chunk","created":1793097986,"model":"qwen/qwen3-32b","provider":"DeepInfra","choices …
  delta {"content":null,"tool_calls":[{"index":0,"id":"call_a30e","type":"function","function":{"name":"lookup_word","arguments":""}}]}
  delta {"content":null,"tool_calls":[{"index":0,"function":{"arguments":"{\"word\": \"ubiquitous\"}"}}]}
  delta {"content":null,"tool_calls":[{"index":1,"id":"call_eee8","type":"function","function":{"name":"lookup_word","arguments":""}}]}
  delta {"content":null,"tool_calls":[{"index":1,"function":{"arguments":"{\"word\": \"mitigate\"}"}}]}
  delta {"content":""}
  delta {"content":""}
data: [DONE]
네트워크 덩어리 100개 · 생각(reasoning) 조각 142줄은 뺐다 · 덩어리 경계에서 잘린 줄 0개


**돌려 보면** 이렇게 나옵니다.

- 응답 한 줄은 `data: {...}` 모양입니다. 앞쪽의 id · model 같은 값은 매 줄 똑같고, 우리가 볼 것은 `choices[0].delta` 하나입니다. `delta` 는 "지난 줄 이후 새로 생긴 부분" 이라는 뜻입니다.
- 도구 요청도 **나눠서** 옵니다. `index: 0` 의 첫 줄에는 id 와 이름(`lookup_word`)만 있고 `arguments` 는 비어 있습니다. 다음 줄에 `{"word": "ubiquitous"}` 가 옵니다. `index: 1` 은 두 번째 도구 요청입니다.
- 생각(reasoning) 줄이 수백 줄 앞에 옵니다. 첫 답 글자가 늦게 나오는 이유입니다. 화면에 찍지 않고 세기만 했습니다.
- 네트워크 덩어리 수와 줄 수가 다릅니다. 덩어리 하나에 줄이 여러 개 들어 오기도 합니다. "잘린 줄" 은 대개 0개지만, 네트워크 상태에 따라 한 줄이 두 덩어리에 나뉘어 올 수 있습니다. 그러면 그 줄은 반쪽만으로는 JSON 으로 읽을 수 없습니다.

In [80]:
// ㉡ 글자로 답할 때: 도구 없이 부르면 답이 토큰 단위로 나눠 온다
await peekSSE([
  SYSTEM,
  { role: "user", content: "meticulous 를 넣은 짧은 영어 예문 하나만 써 줘" },
]);

data: {"id":"gen-1793097991-LBlpmkLGq4yrFsNyEmL3","object":"chat.completion.chunk","created":1793097991,"model":"qwen/qwen3-32b","provider":"DeepInfra","choices …
  delta {"content":"\"She was"}
  delta {"content":" me"}
  delta {"content":"ticulous in"}
  delta {"content":" org"}
  delta {"content":"anizin"}
  delta {"content":"g"}
  delta {"content":" the even"}
  delta {"content":"t, ens"}
  delta {"content":"uring e"}
  delta {"content":"very"}
  delta {"content":" detail "}
  delta {"content":"wa"}
  … delta 20줄 더
data: [DONE]
네트워크 덩어리 236개 · 생각(reasoning) 조각 278줄은 뺐다 · 덩어리 경계에서 잘린 줄 0개


**돌려 보면** 답이 `" me"`, `"ticulous "` 처럼 단어 중간에서도 끊겨 옵니다. 예문은 돌릴 때마다 조금씩 다릅니다. 이어 붙여야 문장이 됩니다.

그래서 SSE 응답을 읽는 쪽이 할 일은 다섯 가지입니다. 이걸 `readStream` 함수 하나로 묶었습니다.

| 하는 일 | 코드 |
| --- | --- |
| ① 덩어리를 줄로 자른다. 덜 온 마지막 줄은 남겨 두었다가 다음 덩어리 앞에 붙인다 | `rest` |
| ② `data: ` 줄만 읽고, 주석 줄과 `[DONE]` 은 건너뛴다 | `startsWith("data: ")` |
| ③ 글자(토큰)는 이어 붙이고, 받는 대로 밖으로 넘긴다 | `content += piece` · `onToken(piece)` |
| ④ 나눠 온 도구 요청은 `index` 별로 id · name · arguments 를 이어 붙인다 | `calls[t.index] ??= …` · `+=` |
| ⑤ 다 모이면 평소 `chat()` 과 **같은 모양의 메시지**로 돌려준다 | `return { role, content, tool_calls }` |

⑤ 덕분에 그래프의 다른 노드는 하나도 고칠 필요가 없습니다. 

In [81]:
// SSE 응답을 읽어 메시지 하나로 모은다. 글자(토큰)는 받는 대로 onToken 에 넘긴다 
async function readStream(
  res: Response,
  onToken: (token: string) => void = () => {},
) {
  if (!res.ok) throw new Error(`LLM 호출 실패 ${res.status}: ${await res.text()}`);

  let content = "",
    rest = "";
  const calls: any[] = [];

  for await (const bytes of res.body!.pipeThrough(new TextDecoderStream())) {
    // ① 덩어리를 줄로 자른다. 덜 온 마지막 줄은 다음 덩어리와 합친다
    const lines = (rest + bytes).split("\n");
    rest = lines.pop()!;
    for (const line of lines) {
      // ② data: 줄만 읽는다. 빈 줄 · 주석 줄 · 끝 표시는 건너뛴다
      if (!line.startsWith("data: ") || line === "data: [DONE]") continue;
      const delta = JSON.parse(line.slice(6)).choices?.[0]?.delta ?? {};

      // ③ 글자(토큰): 이어 붙이고 받는 대로 넘긴다 (맨 앞에 붙는 빈 줄은 뺀다)
      const piece = content ? delta.content : delta.content?.trimStart();
      if (piece) {
        content += piece;
        onToken(piece);
      }

      // ④ 나눠 온 도구 요청: index 별로 id · name · arguments 를 이어 붙인다
      for (const t of delta.tool_calls ?? []) {
        const c = (calls[t.index] ??= {
          id: "",
          type: "function",
          function: { name: "", arguments: "" },
        });
        c.id += t.id ?? "";
        c.function.name += t.function?.name ?? "";
        c.function.arguments += t.function?.arguments ?? "";
      }
    }
  }

  // ⑤ 평소 chat() 과 같은 모양의 메시지로 돌려준다
  return {
    role: "assistant",
    content: content.trim() || null,
    tool_calls: calls.length ? calls : undefined,
  };
}

이제 `chatStream` 은 평소 `chat()` 에 `stream: true` 만 더해 부르고, 응답을 `readStream` 에 넘기면 끝입니다.

In [82]:
// 모델의 글자를 받는 대로 넘겨주는 chat: 평소 chat() 에 stream: true 를 더하고, 응답은 readStream 이 읽는다
async function callModelStream(
  messages: any[],
  onToken: (token: string) => void,
) {
  const res = await fetch(`${BASE_URL}/chat/completions`, {
    method: "POST",
    headers: {
      "Content-Type": "application/json",
      Authorization: `Bearer ${API_KEY}`,
    },
    body: JSON.stringify({
      model: MODEL,
      messages,
      tools,
      ...OPTIONS,
      stream: true,
    }),
  });
  return readStream(res, onToken);
}
const chatStream = asLLM(callModelStream, "chatStream"); // LangSmith 에 모델 호출로 남긴다

이제 tutor 노드가 글자를 받을 때마다 그래프 밖으로 흘려보내면 됩니다. 노드 함수의 두 번째 인자 `config` 에 `writer` 가 있습니다. `config.writer(값)` 으로 보낸 것은 `streamMode: "custom"` 으로 받습니다.

In [83]:
// ④ 토큰: tutor 노드가 받은 글자를 config.writer 로 그래프 밖에 흘려보낸다 → streamMode "custom"
import type { LangGraphRunnableConfig } from "npm:@langchain/langgraph@1.4.18";
async function tutorStream(
  state: typeof State.State,
  config: LangGraphRunnableConfig,
) {
  return {
    messages: [
      await chatStream(state.messages, (token) => config.writer?.(token)),
    ],
  };
}
const streamingAgent = new StateGraph(State)
  .addNode("tutor", tutorStream)
  .addNode("tools", toolsNode)
  .addEdge(START, "tutor")
  .addConditionalEdges("tutor", route, ["tools", END])
  .addEdge("tools", "tutor")
  .compile();
await showGraph(streamingAgent);
// 답이 길수록 차이가 잘 보인다. 끝의 /no_think 는 qwen3 가 답 전에 속으로 생각(reasoning)하는 걸 끄는 스위치
const LONG =
  "IELTS 쓰기 Task 2 에서 결론 문단을 어떻게 쓰는지 다섯 줄로 설명해 줘 /no_think";
// 글자(토큰)를 줄바꿈 없이 바로 찍는다. 노트북에서는 Deno.jupyter 로 보내야 화면에 나온다 (console.log 는 매번 줄을 바꾼다)
const write = (text: string) =>
  Deno.jupyter
    ? Deno.jupyter.broadcast("stream", { name: "stdout", text })
    : Deno.stdout.write(new TextEncoder().encode(text));

t0 = Date.now();
let first = 0;
for await (const [mode, chunk] of await streamingAgent.stream(ask(LONG), {
  streamMode: ["custom"],
})) {
  if (!first) {
    first = Date.now();
    console.log(`첫 글자: ${sec(t0)}`);
  }
  await write(chunk);
}


첫 글자: 0.7초
1. 결론 문단은 주요 논점을 요약하는 데 사용됩니다.  
2. 처음에는 명확한 결론 문장을 시작으로 합니다.  
3. 이어서 본문에서 논의한 주요 아이디어를 간략하게 반복합니다.  
4. 추가적인 생각이나 미래에 대한 예측을 포함할 수 있습니다.  
5. 마지막 문장은 자연스럽게 글을 마무리 짓습니다.

**돌려 보면** 1초쯤에 첫 글자가 나오고, 그다음부터는 글자가 써지는 대로 찍힙니다. 답이 다 써지기까지는 몇 초 더 걸리지만, 보는 사람은 기다리지 않습니다. 이게 stream 의 쓸모입니다.

## ⑤ 모드 여러 개 같이 쓰기

모드는 배열로 여러개를 동시에 사용할수 있습니다.  이번에는 **values 와 custom** 을 같이 받아 보겠습니다. 질문은 ① 의 도구를 쓰는 질문입니다.

- `custom` 값: tutor 노드 **안에서**, 모델이 글자를 쓸 때마다 → 바로 찍는다. 흐르기 시작할 때 `[custom]` 을 한 번 붙인다
- `values` 값: 노드가 **끝날 때마다** State 전체 → messages 개수와 마지막 메시지를 찍는다

받는 값이 `[모드, 값]` 모양이라, `mode` 를 보고 나눠 처리합니다.

In [84]:
// ⑤ 모드 여러 개: values(노드가 끝날 때 State 전체) 와 custom(노드 안에서 보낸 글자) 을 한 번에 받는다
t0 = Date.now();
let streaming = false; // 지금 글자가 흘러나오는 중인가

for await (const [mode, chunk] of await streamingAgent.stream(
  ask(Q + " /no_think"),
  { streamMode: ["values", "custom"] },
)) {
  if (mode === "custom") {
    if (!streaming) await write(`\n[custom · ${sec(t0)}] `); // 글자가 흐르기 시작할 때 한 번만 표시
    streaming = true;
    await write(chunk);
  } else {
    streaming = false;
    console.log(`\n[values · ${sec(t0)}] messages ${chunk.messages.length}개 · 마지막 메시지:`);
    pretty(chunk.messages.at(-1));
  }
}


[values · 0.0초] messages 2개 · 마지막 메시지:
====================== Human Message =======================
이 문장의 ubiquitous 랑 mitigate 를 사전에서 찾아서, 쉬운 말로 바꾼 영어 문장을 만들어 줘: Smartphones are ubiquitous, and schools try to mitigate their effects. /no_think
▶ tools   {"word": "ubiquitous"} · {"word": "mitigate"}

[values · 1.7초] messages 3개 · 마지막 메시지:
======================== Ai Message ========================
Tool Calls: lookup_word({"word": "ubiquitous"}), lookup_word({"word": "mitigate"})

[values · 1.7초] messages 5개 · 마지막 메시지:
======================= Tool Message =======================
{"word":"mitigate","뜻":"완화하다, 줄이다","쉬운말":"reduce"}

[custom · 3.7초] Smartphones are everywhere, and schools try to reduce their effects.
[values · 4.2초] messages 6개 · 마지막 메시지:
======================== Ai Message ========================
Smartphones are everywhere, and schools try to reduce their effects.


**돌려 보면** 두 모드의 값이 시간 순서대로 섞여 들어옵니다.

1. `[values]` messages 2개 — 처음 넣은 입력
2. `[values]` messages 3개 — tutor 의 도구 요청. 도구 요청은 글자가 아니라서 custom 값은 오지 않습니다
3. `[values]` messages 5개 — 사전 결과 두 개
4. `[custom]` 답의 글자가 토큰 단위로 흘러나옵니다
5. `[values]` messages 6개 — 방금 흘러나온 글자가 완성된 메시지 하나로 State 에 들어가 있습니다

화면에는 글자를 흘려 보여 주고(custom), 동시에 State 가 어떻게 쌓이는지도 같이 보는(values) 셈입니다. 받고 싶은 것을 배열에 넣기만 하면 됩니다.